# EPI Recorder — Interactive Demo

> **EPI = Evidence Packaged Infrastructure**  
> A cryptographic flight recorder for AI agents. Every decision, every tool call, every LLM response — sealed in a tamper-evident `.epi` file.

---

### What this demo covers

| # | Feature | What you will see |
|---|---|---|
| 1 | **Recording** | Capture an AI agent's full execution trace as structured steps |
| 2 | **Policy** | Define company rules — what the agent is allowed or forbidden to do |
| 3 | **Fault Intelligence** | 4-pass heuristic + policy-grounded analyzer runs automatically |
| 4 | **Cryptographic Signing** | Ed25519 signature seals the evidence — detects any tampering |
| 5 | **Integrity Verification** | SHA-256 hash of every file, signature check |
| 6 | **Embedded Viewer** | Full visual viewer rendered directly inside this notebook |

### The Scenario: Loan Approval AI Agent Gone Wrong

We simulate a **Loan Approval AI Agent** that makes two critical mistakes:

- Customer **ACC-4821** requests an **$8,500 loan** with only **$5,000 available**
- An error occurs mid-workflow — the agent **ignores it and continues**
- The agent **skips the mandatory credit check** (sequence policy violation)
- The agent approves **$8,500** anyway — exceeding the limit by $3,500

EPI catches all of this automatically.

## Step 1 — Install EPI Recorder

In [ ]:
!pip install epi-recorder -q

import epi_core
print(f"epi-recorder {epi_core.__version__} installed successfully.")

## Step 2 — Build the AI Agent Execution Trace

In production, EPI captures these steps **automatically** via recorder middleware injected into your script.  
Here we build them directly to show the exact data structure being captured.

In [ ]:
import json
import tempfile
import zipfile
from pathlib import Path
from datetime import datetime, timezone

# Working directory for this recording
workspace = Path(tempfile.mkdtemp(prefix="epi_demo_"))
output_path = Path("loan_agent_demo.epi")
print(f"Workspace : {workspace}")
print(f"Output    : {output_path.absolute()}")

def ts(minute, second=0):
    """Generate an ISO timestamp for demo steps."""
    return datetime(2025, 6, 15, 10, minute, second, tzinfo=timezone.utc).isoformat()

# ─── The execution trace ──────────────────────────────────────────────────────
# Each dict is one step. EPI captures these automatically during `epi run`.
# Kinds: session.start/end | tool.call/result | llm.call/response | http.request/response

steps = [
    # Workflow begins
    {"index": 0, "kind": "session.start", "timestamp": ts(0),
     "content": {"workflow": "loan_approval_agent", "session_id": "SESS-7291", "customer_id": "CUST-4821"}},

    # Agent fetches customer account — receives available_funds and credit_limit
    {"index": 1, "kind": "tool.call", "timestamp": ts(1),
     "content": {"tool": "fetch_customer_record", "account_id": "ACC-4821"}},
    {"index": 2, "kind": "tool.result", "timestamp": ts(1, 3),
     "content": {"account_id": "ACC-4821", "customer_name": "Arjun Sharma",
                 "available_funds": 5000.00, "credit_limit": 5000.00, "risk_tier": "B"}},

    # Agent tries to fetch application history — SERVICE IS DOWN (HTTP 503)
    {"index": 3, "kind": "tool.call", "timestamp": ts(2),
     "content": {"tool": "check_application_history", "account_id": "ACC-4821"}},
    {"index": 4, "kind": "tool.result", "timestamp": ts(2, 5),
     "content": {"error": "service_unavailable",
                 "message": "Application history service is down (HTTP 503)",
                 "status_code": 503}},  # <-- PASS 1: Error Continuation will flag this

    # LLM decides on loan — NOTE: no verify_credit_score was ever called
    {"index": 5, "kind": "llm.call", "timestamp": ts(3),
     "content": {"model": "gpt-4o",
                 "prompt": "Customer ACC-4821 requests $8500 loan. Funds: $5000. History service down. Decide."}},
    {"index": 6, "kind": "llm.response", "timestamp": ts(3, 8),
     "content": {"model": "gpt-4o", "decision": "approve",
                 "approved_amount": 8500.00,  # <-- PASS 2: Exceeds $5,000 limit
                 "reasoning": "Proceeding based on risk tier B. History outage is transient.",
                 "confidence": 0.79}},

    # Disbursement — verify_credit_score was never called before this
    {"index": 7, "kind": "tool.call", "timestamp": ts(4),
     "content": {"tool": "process_loan_disbursement",  # <-- PASS 3: R001 sequence violation
                 "account_id": "ACC-4821", "amount": 8500.00, "action": "approve"}},
    {"index": 8, "kind": "tool.result", "timestamp": ts(4, 2),
     "content": {"success": True, "transaction_id": "TXN-98741",
                 "account_id": "ACC-4821", "disbursed_amount": 8500.00}},

    # Sends confirmation — context identifiers vanish from here
    {"index": 9, "kind": "tool.call", "timestamp": ts(5),
     "content": {"tool": "send_customer_notification",
                 "account_id": "ACC-4821", "type": "loan_approved", "amount": 8500.00}},
    {"index": 10, "kind": "tool.result", "timestamp": ts(5, 1),
     "content": {"success": True, "notification_id": "NOTIF-3391", "delivered": True}},

    # Workflow ends — exit code 0, agent thinks it succeeded!
    {"index": 11, "kind": "session.end", "timestamp": ts(6),
     "content": {"exit_code": 0, "duration_seconds": 6.1}},  # <-- PASS 4: CUST-4821 gone
]

(workspace / "steps.jsonl").write_text(
    "\n".join(json.dumps(s) for s in steps), encoding="utf-8"
)
(workspace / "env.json").write_text(
    json.dumps({"python_version": "3.11", "platform": "linux",
                "agent_version": "loan-agent-3.1", "cwd": "/app"}),
    encoding="utf-8"
)

print(f"\nRecorded {len(steps)} steps to steps.jsonl")
print()
kind_icon = {
    "session.start": "[>]", "session.end": "[<]",
    "tool.call": "[T]", "tool.result": "[R]",
    "llm.call": "[L]", "llm.response": "[L]"
}
for s in steps:
    icon = kind_icon.get(s["kind"], "[?]")
    c = s["content"]
    detail = c.get("tool") or c.get("model") or c.get("workflow") or s["kind"]
    flag = "  <-- ERROR" if "error" in c or c.get("status_code", 0) >= 400 else ""
    print(f"  Step {s['index']:2d}  {icon}  {s['kind']:<20}  {detail}{flag}")

## Step 3 — Define the Company Policy

Place `epi_policy.json` in your project root — `epi run` picks it up automatically.  
Four rule types: `constraint_guard`, `sequence_guard`, `threshold_guard`, `prohibition_guard`.

In [ ]:
policy = {
    "system_name": "LoanApprovalSystem",
    "system_version": "3.1",
    "policy_version": "2025-Q2",
    "rules": [
        {
            "id": "R001",
            "name": "Credit Check Required Before Disbursement",
            "severity": "critical",
            "description": "verify_credit_score must be called before process_loan_disbursement.",
            "type": "sequence_guard",
            "required_before": "loan_disbursement",
            "must_call": "verify_credit_score"
        },
        {
            "id": "R002",
            "name": "Loan Must Not Exceed Available Funds",
            "severity": "critical",
            "description": "Approved loan amount must never exceed customer available funds.",
            "type": "constraint_guard",
            "watch_for": ["available_funds", "credit_limit"],
            "violation_if": "approved_amount > available_funds"
        },
        {
            "id": "R003",
            "name": "Loans Above 7500 Require Human Review",
            "severity": "high",
            "description": "Any loan above 7500 requires a human_approval step.",
            "type": "threshold_guard",
            "threshold_value": 7500,
            "threshold_field": "approved_amount",
            "required_action": "human_approval"
        }
    ]
}

(workspace / "epi_policy.json").write_text(json.dumps(policy, indent=2), encoding="utf-8")

print("epi_policy.json written.")
print()
for rule in policy["rules"]:
    print(f"  [{rule['id']}]  {rule['type']:<20}  [{rule['severity'].upper():<8}]  {rule['name']}")

## Step 4 — Fault Intelligence Layer (4-Pass Analysis)

The analyzer runs over `steps.jsonl` using **pure deterministic logic** — no LLM, no network calls.  
It checks the execution trace against the policy rules and general heuristics.

In [ ]:
from epi_core.fault_analyzer import FaultAnalyzer
from epi_core.policy import load_policy

pol = load_policy(search_dir=workspace)
print(f"Policy loaded: {pol.system_name} v{pol.policy_version} ({len(pol.rules)} rules)")

analyzer = FaultAnalyzer(policy=pol)
steps_text = (workspace / "steps.jsonl").read_text(encoding="utf-8")
result = analyzer.analyze(steps_text)

# Save to workspace so it gets sealed into the .epi
(workspace / "analysis.json").write_text(result.to_json(), encoding="utf-8")
(workspace / "policy.json").write_text(json.dumps(policy, indent=2), encoding="utf-8")

sep = "=" * 62
print()
print(sep)
print(f"  FAULT DETECTED : {result.fault_detected}")
print(f"  MODE           : {result.mode}")
print(f"  CONFIDENCE     : {result.confidence}")
print(sep)
print()

sev_label = {"critical": "!! CRITICAL !!", "high": "! HIGH !", "medium": "MEDIUM", "low": "low"}

if result.primary_fault:
    pf = result.primary_fault
    print(f"PRIMARY FAULT  [{sev_label.get(pf.severity, pf.severity)}]")
    print(f"  Type    : {pf.fault_type}")
    if pf.rule_id:
        print(f"  Rule    : {pf.rule_id} -- {pf.rule_name}")
    print(f"  Finding : {pf.plain_english}")
    if pf.fault_chain:
        print(f"  Chain   :")
        for link in pf.fault_chain:
            print(f"    Step {link['step_number']:2d} [{link['role']}] {link['detail']}")
    print()

print(f"SECONDARY FLAGS ({len(result.secondary_flags)} additional findings)")
for i, f in enumerate(result.secondary_flags, 1):
    print(f"  {i}. [{f.severity.upper():<8}] {f.fault_type}")
    print(f"        {f.plain_english[:105]}")

### What each pass detected

| Pass | Detection | Result |
|---|---|---|
| **Pass 1** — Error Continuation | Step 4 returned HTTP 503. Step 5 continued without any error handling | `HEURISTIC` medium |
| **Pass 2** — Constraint Violation | `available_funds = $5,000` at step 2. Agent committed `$8,500` at step 7 — 70% over limit | `HEURISTIC` high |
| **Pass 3** — Sequence Violation | `process_loan_disbursement` executed without `verify_credit_score` — violates **R001** | `POLICY_VIOLATION` critical |
| **Pass 4** — Context Drop | `CUST-4821` and `SESS-7291` established early, vanished from final steps | `HEURISTIC` low |

## Step 5 — Cryptographic Signing and Sealing

In [ ]:
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
from epi_core.trust import sign_manifest
from epi_core.schemas import ManifestModel
from epi_core.container import EPIContainer

# Generate an Ed25519 keypair
# In production: epi run handles this automatically from your stored key
private_key = Ed25519PrivateKey.generate()
public_key_bytes = private_key.public_key().public_bytes_raw()
print(f"Public key  : {public_key_bytes.hex()[:48]}...")

# Build manifest with metadata
manifest = ManifestModel(
    cli_command="epi run loan_agent.py",
    goal="Process loan application for customer ACC-4821",
    notes="Demo recording showcasing fault detection on a policy-violating agent",
    metrics={"loan_amount": 8500.0, "credit_limit": 5000.0, "violation_ratio": 1.7},
    tags=["loan-approval", "demo", "policy-violation"]
)

# Signing function: called by pack() after all files are hashed
# The signature covers: version + file_manifest (every file hash) + metadata
def sign_fn(m):
    return sign_manifest(m, private_key, key_name="demo-key")

# EPIContainer.pack() runs its own internal FaultAnalyzer using Path.cwd() to find
# epi_policy.json. Write it to cwd so pack() produces a policy-grounded analysis.json.
cwd_policy = Path.cwd() / "epi_policy.json"
cwd_policy.write_text(json.dumps(policy, indent=2), encoding="utf-8")

try:
    EPIContainer.pack(workspace, manifest, output_path, signer_function=sign_fn)
finally:
    cwd_policy.unlink(missing_ok=True)  # Remove after packing — keeps cwd clean

size_kb = output_path.stat().st_size / 1024
print(f"\nPacked: {output_path}  ({size_kb:.1f} KB)")
print()

print("Contents of loan_agent_demo.epi:")
with zipfile.ZipFile(output_path, "r") as zf:
    for info in zf.infolist():
        print(f"  {info.filename:<35} {info.file_size:>9,} bytes")

## Step 6 — Verify Integrity

In [ ]:
from epi_core.trust import verify_signature

# Integrity: re-hash every file, compare against manifest.file_manifest
integrity_ok, mismatches = EPIContainer.verify_integrity(output_path)

status = "PASS" if integrity_ok else "FAIL"
print(f"[{status}] File integrity: {'All hashes match' if integrity_ok else str(len(mismatches)) + ' mismatches'}")

# Signature verification
stored = EPIContainer.read_manifest(output_path)
if stored.signature:
    sig_ok, msg = verify_signature(stored, public_key_bytes)
    print(f"[{'PASS' if sig_ok else 'FAIL'}] Ed25519 signature: {msg}")
else:
    print("[INFO] Signature: Not signed")

print()
print("Manifest metadata:")
print(f"  Goal     : {stored.goal}")
print(f"  Tags     : {stored.tags}")
print(f"  Metrics  : {stored.metrics}")
print(f"  Signer   : {stored.public_key[:32] if stored.public_key else 'unsigned'}...")
print(f"  Files    : {len(stored.file_manifest)} files hashed in manifest")

## Step 7 — Tamper Detection Demo

In [ ]:
import shutil

# Create a tampered copy — change 8500 to 1000 to hide the violation
tampered_path = Path("tampered.epi")
shutil.copy2(output_path, tampered_path)

with zipfile.ZipFile(tampered_path, "a") as zf:
    original = zf.read("steps.jsonl").decode("utf-8")
    zf.writestr("steps.jsonl", original.replace("8500.0", "1000.0"))

print("Tampered .epi: modified approved_amount from $8,500 to $1,000 inside the ZIP")
print()

integrity_ok, mismatches = EPIContainer.verify_integrity(tampered_path)
if integrity_ok:
    print("[UNEXPECTED] No tampering detected")
else:
    print(f"[DETECTED] Tampering caught in {len(mismatches)} file(s)!")
    for fname, reason in mismatches.items():
        print(f"  Modified file : {fname}")
        print(f"  Reason        : {reason[:80]}")

tampered_path.unlink()
print()
print("The SHA-256 hash of steps.jsonl in the manifest no longer matches the modified content.")
print("Any edit to any file inside the .epi is immediately detectable.")

## Step 8 — Fault Analysis Report

In [ ]:
with zipfile.ZipFile(output_path, "r") as zf:
    analysis = json.loads(zf.read("analysis.json").decode("utf-8"))

print("analysis.json (sealed inside the .epi — signed with the manifest):")
print()
print(f"  analyzer_version  : {analysis['analyzer_version']}")
print(f"  mode              : {analysis['mode']}")
print(f"  policy_version    : {analysis['policy_version']}")
print(f"  fault_detected    : {analysis['fault_detected']}")
print(f"  confidence        : {analysis['confidence']}")
cov = analysis["coverage"]
print(f"  coverage          : {cov['steps_recorded']} steps, {cov['coverage_percentage']}% with full data")
print()

pf = analysis["primary_fault"]
print("Primary Fault:")
print(f"  type       : {pf['fault_type']}")
print(f"  severity   : {pf['severity'].upper()}")
print(f"  rule       : {pf.get('rule_id')} -- {pf.get('rule_name')}")
print(f"  finding    : {pf['plain_english']}")
print()
print("  Fault chain:")
for link in pf.get("fault_chain", []):
    print(f"    Step {link['step_number']:2d} [{link['role']}]: {link['detail']}")

print()
print("Human review slot (pending sign-off):")
print(f"  {analysis['human_review']}")

## Step 9 — EPI Viewer Embedded in Colab

The interactive viewer is **baked inside the `.epi` file** — fully self-contained HTML + JS + CSS.  
No server. No browser. No external dependencies. Runs right here.

In [ ]:
from IPython.display import display, HTML

# Extract the self-contained viewer from the sealed .epi
with zipfile.ZipFile(output_path, "r") as zf:
    viewer_html = zf.read("viewer.html").decode("utf-8")

print(f"Extracted viewer.html: {len(viewer_html):,} bytes (fully self-contained)")
print("Rendering in Colab...")

# Escape for the srcdoc HTML attribute
# Rule: escape & first (to avoid double-escaping), then escape "
escaped = viewer_html.replace("&", "&amp;").replace('"', "&quot;")

# Build the wrapper — srcdoc renders a complete HTML document in the iframe
# sandbox="allow-scripts allow-same-origin" is required for the JS to run
header = (
    '<div style="border:2px solid #22c55e;border-radius:10px;'
    'overflow:hidden;margin:12px 0;box-shadow:0 4px 16px rgba(0,0,0,.18)">'
    '<div style="background:#16a34a;color:white;padding:10px 16px;'
    'font-family:monospace;font-size:13px;font-weight:bold">'
    'EPI VIEWER  |  loan_agent_demo.epi  |  '
    'Fault: POLICY_VIOLATION [critical]  |  Signed &amp; Verified'
    '</div>'
)
iframe = (
    '<iframe srcdoc="' + escaped + '"'
    ' width="100%" height="840px"'
    ' style="border:none;display:block;background:#0f172a"'
    ' sandbox="allow-scripts allow-same-origin">'
    '</iframe>'
)
footer = '</div>'

display(HTML(header + iframe + footer))

## Step 10 — Using EPI in a Real Project

In production, you just run your script through `epi run` — the recorder instruments everything automatically.

In [ ]:
# Show the CLI experience (documentation only — runs in your terminal)

usage = """
# ─── Record any Python script ────────────────────────────────────
epi run loan_agent.py

# With rich metadata
epi run loan_agent.py \\
    --goal "Process loan for ACC-4821" \\
    --metric loan_amount=8500 \\
    --tag production

# ─── View any recording ──────────────────────────────────────────
epi view loan_agent_20250615_103000      # opens viewer in browser
epi view loan_agent_20250615_103000.epi  # full path also works

# ─── List all recordings ─────────────────────────────────────────
epi ls

# ─── Verify integrity + signature ────────────────────────────────
epi verify loan_agent_20250615_103000

# ─── File association (double-click .epi to open viewer) ─────────
epi associate
"""

print(usage)

captured = """
What EPI captures automatically during `epi run`:

  HTTP requests/responses    via requests + httpx patching
  LLM calls/responses        via OpenAI, Anthropic, Gemini, LangChain hooks
  Environment snapshot       Python version, installed packages, env vars (secrets redacted)
  stdout / stderr            complete output logs captured to the .epi
  Exit code + duration       how the script ended and how long it took
  Fault analysis             runs before sealing — results signed with the manifest
"""
print(captured)

## Summary

| Component | What it does |
|---|---|
| **steps.jsonl** | 12 structured events — every tool call, LLM call, and result |
| **epi_policy.json** | 3 rules defining what the agent may/may not do |
| **FaultAnalyzer** | 4 passes, 7 flags found — primary: POLICY_VIOLATION [critical] from R001 |
| **Ed25519 signature** | Manifest signed after all files hashed — any change breaks the signature |
| **SHA-256 integrity** | Every file hashed — tamper demo showed immediate detection |
| **loan_agent_demo.epi** | Self-contained ZIP: steps + analysis + policy + viewer + manifest |
| **Embedded viewer** | Full interactive browser rendered inside this notebook via iframe srcdoc |

---

**The guarantee**: The `.epi` file is the ground truth.  
The fault analysis is signed alongside the evidence.  
A human reviewer can open any `.epi` and see exactly what the AI did — and whether it followed the rules.